# 0 - Construction de la base de données

Ce notebook présente les fonctionnalités du module `builders` qui contient un ensemble de classes permettant différentes configuration de création de bases de données DuckDB, de tables et d'index associés à ces tables.

### Table des matières

0. [Importation des modules](#section_0)
1. [Importation des données](#section_1)
2. [Construction du schéma](#section_2)
   - [Initialisation de la classe](#section_2_1)
   - [Construction des méta-données](#section_2_2)
   - [Construction des tables de dimensions](#section_2_3)
   - [Construction de la table d'information](#section_2_4)
   - [Création de l'ensemble des tables](#section_2_5)
3. [Construction de la base de données](#section_3)
   - [Construction d'une base de données sans clé primaire](#section_3_1)
     * [Initialisation du builder](#section_3_1_1)
     * [Création du schéma](#section_3_1_2)
     * [Affichage du schéma](#section_3_1_3)
     * [Exemple de requête](#section_3_1_4)
   - [Construction d'une base de données avec clé primaire](#section_3_2)
     * [Création d'une clé primaire composite](section_3_2_1)
     * [Vérification du fonctionnement de la clé primaire dans le cadre de l'insertion de doublons](section_3_2_2)
     * [Condition d'unicité sur la clé primaire lors de la création de la table](section_3_2_3)
4. [Indexation de la base de données](#section_4)
   - [Création d'index](#section_4_1)
   - [Analyse de la table et énumération des index](#section_4_2)

## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import pandas as pd
import numpy as np
import sys
from datetime import datetime, timedelta

# Ajout du chemin
sys.path.append('..')

# Importation des modules ad hoc
from dashboard_template_database.builders.schema import SchemaBuilder
from dashboard_template_database.builders.tables import DuckdbTablesBuilder
from dashboard_template_database.builders.indexer import IndexManager

## 1. Création des données synthétiques <a id="section_1"></a>

In [ ]:
# Définition des paramètres pour les données synthétiques
np.random.seed(42)  # Pour la reproductibilité

# Paramètres
n_rows = 500
start_date = datetime(2023, 1, 1)

# Création des colonnes catégorielles
indicators = ['temperature', 'humidity', 'pressure', 'wind_speed']
countries = ['France', 'Germany', 'Italy', 'Spain', 'Belgium']
kinds = ['forecast', 'observation']
models = ['model_A', 'model_B', 'model_C']
trainings = ['train_v1', 'train_v2']
weeks = list(range(1, 53))
horizons = [1, 3, 7, 14, 30]

# Génération de données uniques pour la clé primaire composite
# Création de toutes les combinaisons possibles puis échantillonnage
data_list = []
for i in range(n_rows):
    # Génération d'une date unique en ajoutant des jours
    date = start_date + timedelta(days=i % 365)
    
    row = {
        'indicator': np.random.choice(indicators),
        'country': np.random.choice(countries),
        'kind': np.random.choice(kinds),
        'model': np.random.choice(models),
        'training': np.random.choice(trainings),
        'week': np.random.choice(weeks),
        'horizon': np.random.choice(horizons),
        'date': date,
        'value': np.random.uniform(10, 100),
        'lower_bound': None if np.random.random() > 0.7 else np.random.uniform(5, 50),
        'upper_bound': None if np.random.random() > 0.7 else np.random.uniform(50, 150),
        'quality_score': np.random.uniform(0, 1),
        'notes': np.random.choice(['OK', 'Warning', None, 'Error'], p=[0.7, 0.15, 0.1, 0.05])
    }
    data_list.append(row)

# Création du DataFrame
df_origin = pd.DataFrame(data_list)

# Suppression des doublons potentiels sur la clé primaire composite
pk_columns = ['indicator', 'country', 'kind', 'model', 'training', 'week', 'horizon', 'date']
df_origin = df_origin.drop_duplicates(subset=pk_columns, keep='first')

# Conversion en datetime
df_origin['date'] = pd.to_datetime(df_origin['date'])

# Définition des labels pour les colonnes
labels = {
    'indicator': 'Indicateur',
    'country': 'Pays',
    'kind': 'Type',
    'model': 'Modèle',
    'training': 'Entraînement',
    'week': 'Semaine',
    'horizon': 'Horizon',
    'date': 'Date',
    'value': 'Valeur',
    'lower_bound': 'Borne inférieure',
    'upper_bound': 'Borne supérieure',
    'quality_score': 'Score de qualité',
    'notes': 'Notes'
}

# Définition des paramètres explicites
CATEGORICAL_THRESHOLD = 10
OUTPUT_PATH = os.path.join('../outputs', 'database_synthetic.db')

# Affichage
print(f"Nombre de lignes générées : {len(df_origin)}")
print(f"Categorical threshold : {CATEGORICAL_THRESHOLD}")
print(f"Output path : {OUTPUT_PATH}")
print(f"\nAperçu des données :")
df_origin.head()

## 2. Construction du schéma <a id="section_2"></a>

### 2.1. Initialisation de la classe <a id="section_2_1"></a>

In [ ]:
# Initialisation du schéma
schema_builder = SchemaBuilder(df=df_origin, categorical_threshold=CATEGORICAL_THRESHOLD)

### 2.2 Construction des méta-données <a id="section_2_2"></a>

In [ ]:
# Construction du jeu de métadonnées
df_metadata = schema_builder.create_metadata_table(column_labels=labels)

df_metadata.head()

### 2.3. Construction des tables de dimensions <a id="section_2_3"></a>

In [ ]:
# Construction des types de dimensions
dimension_tables = schema_builder.create_dimension_tables(column_labels=labels)
dimension_tables['indicator'].head()

### 2.4. Construction de la table d'information <a id="section_2_4"></a>

In [ ]:
# Construction de la table d'informations
df_fact = schema_builder.create_fact_table(column_labels=labels)
df_fact.head()

### 2.5. Création de l'ensemble des tables <a id="section_2_5"></a>

In [ ]:
# Création de l'ensemble des tables du schéma
df_metadata, dimension_tables, df_fact = schema_builder.build(column_labels=labels)

## 3. Construction de la base de données <a id="section_3"></a>

### 3.1. Construction d'une base de données sans clé primaire <a id="section_3_1"></a>

#### 3.1.1. Initialisation du builder <a id="section_3_1_1"></a>

In [ ]:
# Initialisation du builder
builder = DuckdbTablesBuilder(
    df=df_origin, 
    categorical_threshold=CATEGORICAL_THRESHOLD, 
    path=OUTPUT_PATH
)

#### 3.1.2. Création du schéma <a id="section_3_1_2"></a>

In [ ]:
# Construction du schéma duckDB
builder.build_duckdb_schema()

#### 3.1.3. Affichage du schéma <a id="section_3_1_3"></a>

In [ ]:
# Affichage du schéma
builder.display_schema()

#### 3.1.4. Exemple de requête <a id="section_3_1_4"></a>

In [ ]:
# Requête de la table d'information
print(builder.conn.execute("SELECT * FROM dim_model").fetchall())

### 3.2. Construction d'une base de données avec clé primaire <a id="section_3_2"></a>

#### 3.2.1 Création d'une clé primaire composite <a id="section_3_2_1"></a>

In [ ]:
# Définition des clés primaires composites
# Clés adaptées aux données synthétiques (pas de NaN, combinaison unique)
composite_pk = ['indicator', 'country', 'kind', 'model', 'training', 'week', 'horizon', 'date']

# Vérification que les colonnes de clé primaire n'ont pas de NaN
print("Vérification de l'absence de NaN dans les colonnes de clé primaire :")
for col in composite_pk:
    nan_count = df_origin[col].isna().sum()
    print(f"  - {col}: {nan_count} NaN")

# Vérification de l'unicité de la combinaison
duplicates = df_origin.duplicated(subset=composite_pk).sum()
print(f"\nNombre de combinaisons dupliquées : {duplicates}")

# Initialisation du builder avec clés primaires
builder_with_pk = DuckdbTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=composite_pk,
    path=os.path.join('../outputs', 'database_with_pk.db')
)

# Construction du schéma
builder_with_pk.build_duckdb_schema()

#### 3.2.2 Vérification du fonctionnement de la clé primaire dans le cadre de l'insertion de doublons <a id="section_3_2_2"></a>

In [ ]:
# Affichage de la table de métadonnées pour voir les colonnes marquées comme PK
metadata_pk = builder_with_pk.conn.execute(
    "SELECT * FROM metadata WHERE is_primary_key = TRUE"
).fetchdf()
print("Colonnes marquées comme clé primaire:")
print(metadata_pk[['name', 'label', 'is_primary_key']])

# Vérification de la contrainte (tentative d'insertion d'un doublon devrait échouer)
try:
    # Récupération d'une ligne existante
    sample_row = builder_with_pk.conn.execute(
        "SELECT * FROM fact_table LIMIT 1"
    ).fetchdf()

    # Tentative d'insertion d'un doublon (devrait échouer)
    builder_with_pk.conn.execute(f"""
        INSERT INTO fact_table VALUES (
            {sample_row.iloc[0]['indicator']},
            {sample_row.iloc[0]['country']},
            '{sample_row.iloc[0]['date']}',
            999.99,
            {sample_row.iloc[0]['kind']},
            NULL, NULL, NULL, NULL
        )
    """)
    print("ERREUR: L'insertion du doublon aurait dû échouer!")
except Exception as e:
    print("✓ La contrainte de clé primaire fonctionne correctement:")
    print(f"  L'insertion du doublon a été rejetée: {str(e)[:100]}...")

#### 3.2.3 Condition d'unicité sur la clé primaire lors de la création de la table <a id="section_3_2_3"></a>

In [ ]:
# Création d'un DataFrame avec doublons intentionnels
df_with_duplicates = pd.concat([df_origin.head(10), df_origin.head(5)], ignore_index=True)

print(f"DataFrame avec {len(df_with_duplicates)} lignes (dont 5 doublons)")

# Tentative de construction avec clés primaires (devrait échouer)
try:
    builder_bad = DuckdbTablesBuilder(
        df=df_with_duplicates,
        categorical_threshold=CATEGORICAL_THRESHOLD,
        primary_keys=composite_pk
    )
    print("ERREUR: La validation aurait dû échouer!")
except ValueError as e:
    print("✓ Validation réussie:")
    print(f"  {str(e)}")

## 4. Indexation de la base de données <a id="section_4"></a>

### 4.1. Création d'index <a id="section_4_1"></a>

In [ ]:
# Initialisation de l'index manager avec la connexion existante
index_manager = IndexManager(connection=builder_with_pk.conn)

# Création d'un index sur la colonne date
index_manager.create_index(
    table_name='fact_table',
    index_name='idx_fact_date',
    columns=['date']
)

# Création d'un index composite sur country et indicator
index_manager.create_index(
    table_name='fact_table',
    index_name='idx_fact_country_indicator',
    columns=['country', 'indicator']
)



### 4.2. Analyse de la table et énumération des index <a id="section_4_2"></a>

In [ ]:
# Analyse de la table pour mettre à jour les statistiques
index_manager.analyze_table('fact_table')

In [ ]:
# Listing des index créés
indexes = index_manager.list_indexes(table_name='fact_table')

# Affichage
print(f"Nombre d'index sur fact_table: {len(indexes)}")
for idx in range(len(indexes)):
    print(f"  - {indexes.loc[idx, 'index_name']}: {indexes.loc[idx, 'expressions']}")

## A AJOUTER 
- UN CAS DE CONSTRUCTION  SANS TABLE DE DIMENSION (categorical threshold=None)
- UN CAS AVEC DES DONNEES PARQUET, INDEXATION ET PARTITIONNEMENT
